# VecDB Financial Accounts BYOV Quickstart
Generate embeddings client-side, load vectors, and query with metadata filters.

### What you'll do
1. Environment setup
2. Configure connection
3. Create a client
4. Quick health check
5. Create table for uploaded embeddings
6. Generate embeddings and upsert vectors
7. Filtered semantic search
8. Inspect recommended offer
9. Cleanup demo artifacts


## 1. Environment Setup
- Oracle Autonomous AI Vector Database with permissions to create vector tables.
- `.env` file with `VECDB_REST_URL`, credentials, and any optional proxy settings.
- Python 3.10+ virtual environment activated.


In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas


## 2. Configure Connection
Load environment variables and define convenience constants used throughout the notebook.


In [ ]:
from dotenv import load_dotenv

load_dotenv()
ACCOUNT_TABLE = "FINANCE_ACCOUNTS_BYOV"
N_RESULTS = 6
# MODEL_NAME = "all_MiniLM_L12_v2"
MODEL_NAME = "TEXT_EMBED_MODEL"


## 3. Create Client
Instantiate `OracleVecDB` using environment configuration so it can be reused across queries.


In [ ]:
import os
from oracle_vecdb import OracleVecDB, Configuration

resolved_host = os.getenv("VECDB_REST_URL")
resolved_user = os.getenv("VECDB_USER") or os.getenv("VECDB_USERNAME")
resolved_password = os.getenv("VECDB_PASSWORD")
resolved_access_token = os.getenv("VECDB_ACCESS_TOKEN")

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
# This can be used with local AI Database with ORDS using self signed certs
if os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true":
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print(f"SDK client ready, using REST endpoint: {config.rest_url}")
print('Auth method:', auth_method)


## 4. Quick Health Check
Ping the service and inspect database level statistics before running product searches.


In [ ]:
db_stats = vecdb.describe_vector_database()
print(
    f"Models: {db_stats.total_models} | Tables: {db_stats.total_tables} | Vectors: {db_stats.total_vectors:,}"
)


## 5. Create Table for Uploaded Embeddings
Provision a dense vector table that will store embeddings generated in this notebook.


In [ ]:
# Clean slate for demo reruns
existing_tables = vecdb.list_vector_tables().items or []
if any(t.table_name == ACCOUNT_TABLE for t in existing_tables):
    vecdb.drop_vector_table(name=ACCOUNT_TABLE)
    print(f"Dropped existing table: {ACCOUNT_TABLE}")

vecdb.create_vector_table(
    name=ACCOUNT_TABLE,
    table_params={"auto_generate_id":False},
    annotations={
        "ACCOUNT_NAME": "string",
        "ACCOUNT_CATEGORY": "string",
        "INTEREST_APY": "float",
        "MIN_BALANCE": "float",
        "MONTHLY_FEE": "float",
        "ELIGIBILITY": "string",
        "FEATURE_TAGS": "string",
        "ACCOUNT_SUMMARY": "string",
    },
    index_params={
        "metadata_index_params": {
            "auto_index": True,
            "include_paths": ["ACCOUNT_CATEGORY", "ELIGIBILITY", "MONTHLY_FEE"],
        },
    },
)


## 6. Setup Sample Data and Generate Embeddings
Leverage a hosted model to create embeddings locally

In [ ]:
accounts_dataset = [
    {
        "id": "ACC-100",
        "text": "Student Advantage Savings with 4.25% APY, no minimum balance, and budgeting workshops.",
        "metadata": {
            "ACCOUNT_NAME": "Student Advantage Savings",
            "ACCOUNT_CATEGORY": "Savings",
            "INTEREST_APY": 4.25,
            "MIN_BALANCE": 0.0,
            "MONTHLY_FEE": 0.0,
            "ELIGIBILITY": "Student",
            "FEATURE_TAGS": "Budgeting Tools | Auto-Save | Mobile App",
            "ACCOUNT_SUMMARY": "High-yield savings tailored for students with financial literacy resources.",
        },
    },
    {
        "id": "ACC-200",
        "text": "Premier Checking with concierge support, waived fees over $5k balance, and travel perks.",
        "metadata": {
            "ACCOUNT_NAME": "Premier Checking",
            "ACCOUNT_CATEGORY": "Checking",
            "INTEREST_APY": 0.45,
            "MIN_BALANCE": 5000.0,
            "MONTHLY_FEE": 25.0,
            "ELIGIBILITY": "Premium",
            "FEATURE_TAGS": "Travel Insurance | Concierge",
            "ACCOUNT_SUMMARY": "A premium account with lifestyle perks for affluent customers.",
        },
    },
    {
        "id": "ACC-300",
        "text": "Everyday Rewards Checking offering cashback on debit purchases and bill pay automation.",
        "metadata": {
            "ACCOUNT_NAME": "Everyday Rewards Checking",
            "ACCOUNT_CATEGORY": "Checking",
            "INTEREST_APY": 0.2,
            "MIN_BALANCE": 100.0,
            "MONTHLY_FEE": 5.0,
            "ELIGIBILITY": "All Customers",
            "FEATURE_TAGS": "Cashback | Bill Pay | Mobile Alerts",
            "ACCOUNT_SUMMARY": "Everyday checking with cashback incentives and digital tooling.",
        },
    },
    {
        "id": "ACC-400",
        "text": "High Yield Goal Savings with 5.1% APY, goal tracking, and family sharing.",
        "metadata": {
            "ACCOUNT_NAME": "High Yield Goal Savings",
            "ACCOUNT_CATEGORY": "Savings",
            "INTEREST_APY": 5.1,
            "MIN_BALANCE": 1000.0,
            "MONTHLY_FEE": 0.0,
            "ELIGIBILITY": "All Customers",
            "FEATURE_TAGS": "Goal Tracking | Family Sharing | Rate Boost",
            "ACCOUNT_SUMMARY": "Goal-based savings with tools for households and savers focused on returns.",
        },
    },
]

embedded_vectors = []
for record in accounts_dataset:
    response = vecdb.generate_embedding(model_name=MODEL_NAME, inputs=record["text"])
    vector = response.data[0].embedding
    embedded_vectors.append(
        {
            "id": record["id"],
            "dense_vector": vector,
            "metadata": record["metadata"],
        }
    )

vecdb.upsert_vectors(table_name=ACCOUNT_TABLE, vectors=embedded_vectors)
print(f"Inserted {len(embedded_vectors)} account vectors")


## 7. Perform a simple semantic query
Use the generated embeddings to retrieve the closest matching accounts.


In [ ]:
import pandas as pd
from textwrap import shorten

def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def first_query_item(response):
    items = query_items(response)
    return items[0] if items else None


preview_prompt = "Find a bank account for a new graduate"
preview_embedding = vecdb.generate_embedding(model_name=MODEL_NAME, inputs=preview_prompt)
preview_vector = preview_embedding.data[0].embedding

preview_results = vecdb.query(
    table_name=ACCOUNT_TABLE,
    query_by={"vector": preview_vector},
    top_k=N_RESULTS,
    include_vectors=False,
)

rows = []
for item in query_items(preview_results):
    meta = result_metadata(item)
    rows.append(
        {
            "Product": meta.get("ACCOUNT_NAME"),
            "Category": meta.get("ACCOUNT_CATEGORY"),
            "APY": meta.get("INTEREST_APY"),
            "Min Balance": meta.get("MIN_BALANCE"),
            "Monthly Fee": meta.get("MONTHLY_FEE"),
            "Eligibility": meta.get("ELIGIBILITY"),
            "Highlights": shorten(str(meta.get("FEATURE_TAGS")), width=80, placeholder="…"),
        }
    )

pd.DataFrame(rows)


## 8. Filtered Semantic Search
Blend semantic similarity and metadata filtering to surface the most relevant account for a customer scenario.


Here we apply a composite filter using `$and`, `$eq`, and `$lte` operators.


In [ ]:
filters = {
    "$and": [
        {"ACCOUNT_CATEGORY": {"$eq": "Savings"}},
        {"ELIGIBILITY": {"$eq": "Student"}},
        {"MONTHLY_FEE": {"$lte": 5}},
    ]
}

search_text = "high-yield savings with student perks and mobile budgeting tools"
search_embedding = vecdb.generate_embedding(model_name=MODEL_NAME, inputs=search_text)
search_vector = search_embedding.data[0].embedding

filtered_results = vecdb.query(
    table_name=ACCOUNT_TABLE,
    query_by={"vector": search_vector},
    filters=filters,
    top_k=3,
    include_vectors=False,
)

recommendations = []
for item in query_items(filtered_results):
    meta = result_metadata(item)
    recommendations.append(
        {
            "Product": meta.get("ACCOUNT_NAME"),
            "APY": meta.get("INTEREST_APY"),
            "Min Balance": meta.get("MIN_BALANCE"),
            "Monthly Fee": meta.get("MONTHLY_FEE"),
            "Features": meta.get("FEATURE_TAGS"),
            "Distance": round(result_distance(item) or 0.0, 4),
        }
    )

pd.DataFrame(recommendations)


## 9. Inspect Recommended Offer
Inspect the top match and present the full metadata payload for downstream approval flows.


In [ ]:
from pprint import pprint

top_hit = first_query_item(filtered_results)
if top_hit:
    top_metadata = result_metadata(top_hit)
    print(f"Suggested account: {top_metadata.get('ACCOUNT_NAME')}")
    pprint(top_metadata)
else:
    print("No accounts matched the current filters. Adjust search_text or filters.")


## 10. Cleanup Demo Artifacts
Remove the demo table to leave the environment tidy for subsequent runs.


In [ ]:
vecdb.drop_vector_table(name=ACCOUNT_TABLE)
print(f"Dropped demo table: {ACCOUNT_TABLE}")
